# SupplyMind Training Run

This notebook is the compact, judge-runnable training path for SupplyMind: environment smoke test → SFT warm-start → GRPO from SFT → held-out evaluation.

Default settings train the **center** role on the easy task so the notebook can run quickly. Change `ROLE` to `"warehouse"` or `TASK_ID` to `"v2_train_medium"` / `"v2_train_hard"` for a larger run.

## 1. Setup

In [ ]:
!pip -q install "torch" "transformers>=4.45.0" "trl>=0.12.0" "peft>=0.13.0" accelerate datasets bitsandbytes huggingface_hub pydantic pyyaml matplotlib

In [ ]:
import os
from pathlib import Path

if not Path("supplymind").exists():
    !git clone -q https://huggingface.co/spaces/rishavutk/supplymind supplymind

%cd /content/supplymind
!pip -q install -e .

In [ ]:
from huggingface_hub import notebook_login, whoami

notebook_login()
HF_NAMESPACE = whoami()["name"]
print("Using HF namespace:", HF_NAMESPACE)

## 2. Controls

Change only these values for quick variants. `ROLE` controls which policy is trained; `TASK_ID` controls easy/medium/hard world generation.

In [ ]:
ROLE = "center"  # "center" or "warehouse"
TASK_ID = "v2_train_easy"  # also: "v2_train_medium", "v2_train_hard"
TRAIN_SEEDS = "101,113,127"
EVAL_SEEDS = "131,149,163"
SFT_STEPS = 20
GRPO_STEPS = 20
MAX_COMPLETION_LENGTH = 256

SFT_ADAPTER_ID = f"{HF_NAMESPACE}/supplymind-{ROLE}-qwen-0.5b-sft-notebook"
GRPO_ADAPTER_ID = f"{HF_NAMESPACE}/supplymind-{ROLE}-qwen-0.5b-grpo-notebook"

print({
    "role": ROLE,
    "task_id": TASK_ID,
    "train_seeds": TRAIN_SEEDS,
    "eval_seeds": EVAL_SEEDS,
    "sft_adapter": SFT_ADAPTER_ID,
    "grpo_adapter": GRPO_ADAPTER_ID,
})

## 3. Environment Smoke Test

In [ ]:
import json
import os
import sys

sys.path.insert(0, "/content/supplymind/src")
os.environ["SUPPLYMIND_REWARD_CONFIG"] = "/content/supplymind/configs/supplymind_v2_rewards.yaml"

from supplymind_env_v2.environment import V2SupplyMindEnv
from supplymind_env_v2.models import V2JointAction

env = V2SupplyMindEnv(default_task_id=TASK_ID)
obs = env.reset_internal(TASK_ID, 131)
print("observation keys:", sorted(obs.model_dump(mode="json").keys()))
print("round:", obs.round_index, "warehouses:", list(obs.warehouses.keys()))

empty_action = {
    "warehouse_actions": {},
    "central_action": {
        "central_procurements": [],
        "central_liquidations": [],
        "central_replenishments": [],
        "inventory_transfer_proposals": [],
        "offer_matches": [],
    },
}
result = env.step(V2JointAction.model_validate(empty_action))
print("step reward:", result.reward.step_reward)
print("done:", result.done)
print("info keys:", sorted(result.info.keys()))

## 4. SFT Warm-Start

SFT teaches the model the action JSON shape and a reasonable heuristic policy. For the warehouse role, the notebook enables the conservative SFT flag to reduce invalid or overactive actions.

In [ ]:
warehouse_flags = "--warehouse-conservative-sft --warehouse-signal-limit 2" if ROLE == "warehouse" else ""

!python scripts/hf_sft_supplymind_roles.py \
  --role {ROLE} \
  --task-id {TASK_ID} \
  --seeds {TRAIN_SEEDS} \
  --max-steps {SFT_STEPS} \
  --hub-model-id {SFT_ADAPTER_ID} \
  --output-dir outputs/{ROLE}-sft-notebook \
  {warehouse_flags}

## 5. GRPO From SFT

The GRPO script routes reward by role: center updates from center reward deltas, warehouse updates from warehouse reward deltas, while global reward is logged for audit.

In [ ]:
!python scripts/hf_train_supplymind_roles.py \
  --role {ROLE} \
  --task-id {TASK_ID} \
  --seeds {TRAIN_SEEDS} \
  --max-steps {GRPO_STEPS} \
  --max-completion-length {MAX_COMPLETION_LENGTH} \
  --init-adapter-id {SFT_ADAPTER_ID} \
  --hub-model-id {GRPO_ADAPTER_ID} \
  --output-dir outputs/{ROLE}-grpo-notebook

## 6. Held-Out Evaluation

Evaluate base Qwen, SFT, and GRPO on held-out seeds. The role score is the training target; global score is the environment-level audit metric.

In [ ]:
!python scripts/hf_eval_supplymind_adapters.py \
  --role {ROLE} \
  --task-id {TASK_ID} \
  --seeds {EVAL_SEEDS} \
  --sft-adapter-id {SFT_ADAPTER_ID} \
  --grpo-adapter-id {GRPO_ADAPTER_ID} \
  --max-new-tokens {MAX_COMPLETION_LENGTH} | tee outputs/{ROLE}-eval-notebook.log

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

log_path = Path(f"outputs/{ROLE}-eval-notebook.log")
rows = []
for line in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
    line = line.strip()
    if not line.startswith("{"):
        continue
    try:
        payload = json.loads(line)
    except json.JSONDecodeError:
        continue
    if payload.get("message") == "eval_done":
        evaluations = {key: payload[key] for key in ("base", "sft", "grpo") if key in payload}
        for label, item in evaluations.items():
            role_score_key = "mean_center_role_score" if ROLE == "center" else "mean_warehouse_role_score"
            rows.append({
                "policy": label,
                "global_score": item.get("mean_global_score"),
                "role_score": item.get(role_score_key),
                "raw_reward": item.get("mean_raw_reward"),
                "invalid_payloads": item.get("invalid_payloads"),
                "invalid_actions": item.get("invalid_actions"),
            })

df = pd.DataFrame(rows)
display(df)

if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    df.plot.bar(x="policy", y="role_score", ax=axes[0], legend=False, color="#2563eb")
    axes[0].set_title(f"{ROLE} role score")
    axes[0].set_ylim(0, 1)
    df.plot.bar(x="policy", y=["invalid_payloads", "invalid_actions"], ax=axes[1], color=["#dc2626", "#f59e0b"])
    axes[1].set_title("Invalid outputs")
    plt.tight_layout()
    plt.show()

## Rerun For The Other Role

To train warehouses instead of center, change `ROLE = "warehouse"` in the controls cell and rerun sections 4-6. To try larger worlds, change `TASK_ID` to `v2_train_medium` or `v2_train_hard`.